# LoRA and PEFT

**Track:** Fine-tuning and alignment. **Level:** Advanced. **CPU:** yes.

## Goals

- Freeze a linear layer and train a low-rank update `x @ A @ B`.
- Check that merging the update into the base weight reproduces the forward pass.
- Attach Hugging Face PEFT to a randomly initialized GPT-2 that is not downloaded.

## Sources and license

The update is the factorization in Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models* (2021). The implementation is original. The second half calls Hugging Face PEFT (Apache-2.0) on a `GPT2LMHeadModel` built from a config, so the lab does not download a checkpoint. PEFT's own source is not copied.

## Concepts

Full fine-tuning updates every weight. LoRA leaves `W` frozen and learns `ΔW = (α / r) B A`, with `A` shaped `(in, r)` and `B` shaped `(r, out)` in this notebook's layout. `r` is the rank. If `r` is much smaller than `min(in, out)`, the number of trained numbers drops by about that factor. Merging adds `ΔW` into `W` so inference does not pay for the extra matmul. PEFT does the same attachment to the modules you name. Here that module is GPT-2's `c_attn`.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'torch', 'transformers', 'peft'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

import torch
import torch.nn as nn
import torch.nn.functional as F
from peft import LoraConfig, TaskType, get_peft_model
from transformers import GPT2Config, GPT2LMHeadModel

from labs._shared.seed import seed_everything

seed_everything(0)
torch.manual_seed(0)


## Guided exercise: LoRA from scratch

The base weight is frozen at zero, so the low-rank factors have to fit the target by themselves. The target map has rank at most 4, and `r` is 4, so a fit is possible. After training, `A.grad` exists and `base.weight.requires_grad` is false.


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=4, alpha=8.0):
        super().__init__()
        self.base = nn.Linear(in_features, out_features)
        self.base.weight.requires_grad_(False)
        self.base.bias.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(in_features, r) * 0.02)
        self.B = nn.Parameter(torch.zeros(r, out_features))
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A @ self.B) * self.scale

    def merged_weight(self):
        # nn.Linear stores W as (out, in). x @ A @ B adds scale * (B.T @ A.T).
        return self.base.weight.detach() + self.scale * (self.B.detach().T @ self.A.detach().T)


torch.manual_seed(0)
x = torch.randn(256, 32)
# The target map has rank 4, which matches r below and is out of reach for r=1.
y = x @ (torch.randn(32, 4) @ torch.randn(4, 32))
layer = LoRALinear(32, 32, r=4, alpha=8.0)
with torch.no_grad():
    layer.base.weight.zero_()
    layer.base.bias.zero_()
opt = torch.optim.Adam([layer.A, layer.B], lr=0.08)
start = float(F.mse_loss(layer(x), y).detach())
for _ in range(120):
    loss = F.mse_loss(layer(x), y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
print("mse", round(start, 3), "->", round(float(loss.detach()), 3))
assert float(loss.detach()) < start * 0.2
assert layer.base.weight.requires_grad is False
assert layer.A.grad is not None

merged = nn.Linear(32, 32)
with torch.no_grad():
    merged.weight.copy_(layer.merged_weight())
    merged.bias.copy_(layer.base.bias)
probe = torch.randn(8, 32)
assert torch.allclose(merged(probe), layer(probe), atol=1e-5)
trainable = layer.A.numel() + layer.B.numel()
full = layer.base.weight.numel()
print("lora params", trainable, "full params", full)
assert trainable < full


## Guided exercise: PEFT on a tiny GPT-2

The model is constructed from `GPT2Config`. Nothing is downloaded. `target_modules=["c_attn"]` is the packed query/key/value projection. Trainable parameters must be a minority of the total. One optimizer step has to produce a finite loss. `bos_token_id` is set inside the vocab so the config warning stays quiet.


In [ ]:
config = GPT2Config(
    vocab_size=64,
    n_positions=32,
    n_embd=32,
    n_layer=2,
    n_head=2,
    n_inner=64,
    bos_token_id=0,
    eos_token_id=0,
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)
base = GPT2LMHeadModel(config)
peft_model = get_peft_model(
    base,
    LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=4,
        lora_alpha=8,
        lora_dropout=0.0,
        target_modules=["c_attn"],
    ),
)
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in peft_model.parameters())
print(f"trainable {trainable} / {total}")
assert 0 < trainable < total * 0.5
ids = torch.randint(0, 64, (8, 16))
opt = torch.optim.AdamW((p for p in peft_model.parameters() if p.requires_grad), lr=1e-3)
peft_model.train()
out = peft_model(input_ids=ids, labels=ids)
out.loss.backward()
opt.step()
print("one-step loss", round(float(out.loss.detach()), 3))
assert torch.isfinite(out.loss).item()
print("LAB OK")


## Your turn

Print the trainable parameter count of the from-scratch layer at `r=1` and at `r=8`. The count is `r * (in + out)` plus the frozen base, which does not change.

## Solution


In [ ]:
def trainable_count(rank: int) -> int:
    module = LoRALinear(16, 4, r=rank)
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

low, high = trainable_count(1), trainable_count(8)
print("r=1", low, "r=8", high)
assert high == 8 * low


## Beyond the course

Fit the same target with `r=1` and with `r=4` for the same number of steps, from the same kind of initialization. `r=1` cannot represent a rank-4 map. Its final loss should be worse. This is the rank ablation on a problem where the answer is known.


In [ ]:
def fit(rank: int) -> float:
    torch.manual_seed(0)
    module = LoRALinear(32, 32, r=rank, alpha=float(rank))
    with torch.no_grad():
        module.base.weight.zero_()
        module.base.bias.zero_()
    optimizer = torch.optim.Adam([module.A, module.B], lr=0.08)
    last = None
    for _ in range(120):
        last = F.mse_loss(module(x), y)
        optimizer.zero_grad(set_to_none=True)
        last.backward()
        optimizer.step()
    return float(last.detach())

loss_r1, loss_r4 = fit(1), fit(4)
print("r=1", round(loss_r1, 3), "r=4", round(loss_r4, 3))
assert loss_r4 < loss_r1


## Research questions

1. Merging is exact for this linear layer. What becomes inexact if the forward pass applied dropout to the LoRA branch during training?
2. PEFT trained `c_attn` only. Which GPT-2 tensors are still frozen, and which of them would you unfreeze for a new output vocabulary?
3. The rank ablation used a target of rank 4. How would you estimate the rank you need when the target is a language-model loss and you cannot see `W`?

## Further reading

- Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models* (2021).
- Dettmers et al., *QLoRA: Efficient Finetuning of Quantized Language Models* (2023).
- Hugging Face PEFT documentation (Apache-2.0). https://github.com/huggingface/peft
